# Student Mental Health: EDA, Preprocessing, Model Training, and Evaluation

Dataset: `data/Education_Dataset.xlsx`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os
%matplotlib inline


In [ ]:
data_path = 'data/Education_Dataset.xlsx'
df = pd.read_excel(data_path)
df.head()


## EDA: Overview and Missing Values


In [ ]:
df.shape, df.dtypes.value_counts()


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
fig, axes = plt.subplots(nrows=min(4, len(numeric_cols)), ncols=1, figsize=(8, 4*min(4, len(numeric_cols))))
if len(numeric_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, numeric_cols[:4]):
    sns.histplot(df[col].dropna(), kde=True, ax=ax)
    ax.set_title(f'Distribution: {col}')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df[numeric_cols].corr(), cmap='coolwarm', annot=False)
plt.title('Correlation Heatmap (Numeric)')
plt.show()


## Preprocessing
- Drop `Student_ID` if present
- Impute missing values (mode for categorical, mean for numeric)
- One-hot encode categoricals
- Standardize features


In [ ]:
work = df.copy()
if 'Student_ID' in work.columns:
    work = work.drop('Student_ID', axis=1)
for col in work.columns:
    if work[col].dtype == 'object':
        work[col] = work[col].fillna(work[col].mode().iloc[0])
    else:
        work[col] = work[col].fillna(work[col].mean())
cat_cols = ['Gender', 'Course', 'Living_Type', 'Club_Participation', 'Counseling_Access']
present_cat = [c for c in cat_cols if c in work.columns]
work = pd.get_dummies(work, columns=present_cat, drop_first=True)
X = work.drop('CGPA', axis=1)
y = work['CGPA']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
feature_names = X.columns
len(feature_names)


## Model Training and Evaluation


In [ ]:
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mse, r2


In [ ]:
imp = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
imp.head(15)


In [ ]:
os.makedirs('model', exist_ok=True)
joblib.dump(model, 'model/random_forest_regressor.joblib')
joblib.dump(scaler, 'model/scaler.joblib')
'saved'
